In [35]:
from langchain_community.document_loaders import PyPDFLoader,TextLoader,UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import os
from dotenv import load_dotenv
import anthropic

In [3]:
pdfloader = PyPDFLoader('Attention is all you need.pdf')
textloader = TextLoader('anthropic-core-views-on-ai-safety.txt')
mdloader = UnstructuredMarkdownLoader('transformer_explainer.md')

In [5]:
pdf_doc = pdfloader.load()
text_doc = textloader.load()
md_doc = mdloader.load()

Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 45 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 86 0 (offset 0)
Ignoring wrong pointing object 130 0 (offset 0)
Ignoring wrong pointing object 1403 0 (offset 0)
Ignoring wrong pointing object 2498 0 (offset 0)
Ignoring wrong pointing object 3381 0 (offset 0)
Ignoring wrong pointing object 4389 0 (offset 0)


In [7]:
len(pdf_doc)
pdf_doc[0].page_content[:100]

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and'

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 150,
    keep_separator=True
    )

In [10]:
docs = splitter.split_documents(pdf_doc+text_doc+md_doc)

In [12]:
print(len(docs))
print(docs[0].page_content[:100])
print(docs[0].metadata)

83
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and
{'producer': 'macOS Version 26.5.1 (Build 25F80) Quartz PDFContext, AppendMode 1.1', 'creator': 'Safari', 'creationdate': "D:20260716001456Z00'00'", 'author': 'Swayam Mestry', 'moddate': "D:20260717203744Z00'00'", 'title': 'https://arxiv.org/pdf/1706.03762', 'source': 'Attention is all you need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [16]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
embeddings = model.encode([docs[i].page_content for i in range(len(docs))])
print(embeddings.shape[0])

83


In [23]:
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [43]:
query = model.encode(["what is the capital of france?"])

In [44]:
distances,indices = index.search(query,2)

In [45]:
print(distances)
print(indices)

[[1.7201412 1.7340226]]
[[ 1 47]]


In [32]:
print(docs[81].page_content)

of individual heads is mixed then followed by a computing feed-forward layer. residual connections are commonly used after each step to prevent vanishing gradients. dropouts can be used to prevent overfitting. Normalization is used before residuals to scale outputs. transformers can have multiple such layers in sequence, called "blocks". the output of the last layers gets a final normalization Decoder has 'masking' meaning that we hide future tokens' weights so that the model does not learn from then since the goal of this is for each token to predict its next token sequentially with information from all previous tokens and itself. masking is done right before softmax setting upper triangular matrix to -inf so that softmax converts it to zero while still keeping total =1. cross-attention


In [34]:
load_dotenv()
api_key = os.getenv("API_KEY")

In [36]:
client = anthropic.Anthropic(api_key=api_key)
response = client.messages.create(
    model = 'claude-haiku-4-5-20251001',
    max_tokens = 10,
    messages = [
        {'role':'user','content':'say hello'}
    ]
)

In [37]:
print(response.content[0].text)

Hello! 👋 How can I help


In [46]:
response = client.messages.create(
    model = 'claude-haiku-4-5-20251001',
    max_tokens = 100,
    system = f'You will only answer using context from {docs[1].page_content} and {docs[47].page_content} and say not found if you cannot find required answer in this',
    messages = [
        {'role':'user','content':'what is the capital of france?'}
    ]
)

In [47]:
print(response.content[0].text)

I cannot answer this question because it falls outside the context provided to me. I have been instructed to only answer using information from the document about the Transformer neural network architecture, which discusses convolutional neural networks, encoders, decoders, attention mechanisms, and machine translation tasks.

The question about the capital of France is not found in the provided context, so I must say: **not found**
